# OpenPlaque — TotalSegmentator validation on source CCTA

This notebook is intentionally isolated from the earlier RCA heuristic experiments.

It starts from the solid OpenPlaque `main` branch and uses only its stable study-loading code to:
1. load source CCTA **series 7**,
2. run TotalSegmentator's open `aorta` segmentation,
3. attempt TotalSegmentator's licensed `coronary_arteries` segmentation,
4. render static validation overlays.

No centerline is created in this notebook.

Use **Runtime → Run all**. A GPU runtime is strongly recommended.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Fresh clone of this clean validation branch and required packages.
!rm -rf /content/OpenPlaque
!git clone -q --branch totalsegmentator-validation-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque

%pip -q install TotalSegmentator pydicom SimpleITK nibabel scipy matplotlib pandas

import os, sys, time, shutil, subprocess, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from openplaque.study import OpenPlaqueStudy

print('OpenPlaque source:', SRC)
print('Python:', sys.version.split()[0])

try:
    import torch
    DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'
except Exception:
    DEVICE = 'cpu'
print('TotalSegmentator device:', DEVICE)


## Load source CCTA series 7

`Full_DICOM.zip` is staged on local Colab storage before extraction so repeated DICOM access does not run directly from Drive.


In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_totalseg'
SOURCE_SERIES = 7

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying {DRIVE_ZIP.name} ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...', flush=True)
    t = time.time()
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copy complete in {time.time()-t:.1f}s')
else:
    print('Local ZIP already staged.')

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
print('Extracting/scanning DICOM locally...', flush=True)
t = time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'DICOM scan complete in {time.time()-t:.1f}s; {len(study.series)} series found.')

source_img, source, source_files = study.load_series(SOURCE_SERIES)
print('Loaded source series:', SOURCE_SERIES)
print('Shape z,y,x:', source.shape)
print('Spacing x,y,z mm:', source_img.GetSpacing())
print('Origin:', source_img.GetOrigin())
print('Direction:', source_img.GetDirection())

INPUT_NII = Path('/content/source_series7.nii.gz')
sitk.WriteImage(source_img, str(INPUT_NII))
print('Wrote:', INPUT_NII)


## Confirm TotalSegmentator capabilities

The ordinary `total` task contains an `aorta` class. The dedicated `coronary_arteries` task may require a TotalSegmentator license.

If you have a license, you may store it in Colab Secrets under the name **`TOTALSEG_LICENSE`**. The notebook never prints the secret. A previously configured TotalSegmentator license also works.


In [ ]:
def run_capture(cmd):
    print('$', ' '.join(cmd))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout[-6000:])
    return p.returncode, p.stdout

# Introspection only; this downloads no model weights.
for task in ('total', 'coronary_arteries'):
    rc, out = run_capture(['totalseg_info', '--classes', '-ta', task])
    if rc != 0:
        print(f'Warning: could not introspect task {task}.')


## Run aorta segmentation

This uses TotalSegmentator's normal `total` task but requests only the `aorta` ROI.


In [ ]:
AORTA_DIR = Path('/content/totalseg_aorta')
shutil.rmtree(AORTA_DIR, ignore_errors=True)
AORTA_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    'TotalSegmentator',
    '-i', str(INPUT_NII),
    '-o', str(AORTA_DIR),
    '-ta', 'total',
    '-rs', 'aorta',
    '--device', DEVICE,
]
print('Running aorta segmentation. First use may download model weights...', flush=True)
t = time.time()
rc_aorta, out_aorta = run_capture(cmd)
print(f'Aorta runtime: {(time.time()-t)/60:.1f} min')
if rc_aorta != 0:
    raise RuntimeError('TotalSegmentator aorta segmentation failed. See output above.')


## Attempt coronary-artery segmentation

The current TotalSegmentator `coronary_arteries` model may be licensed. This cell:
- uses a Colab secret named `TOTALSEG_LICENSE` if present,
- otherwise tries any TotalSegmentator license already configured in the runtime,
- and if unavailable, **does not crash the notebook**. The aorta validation still completes.


In [ ]:
COR_DIR = Path('/content/totalseg_coronary')
shutil.rmtree(COR_DIR, ignore_errors=True)
COR_DIR.mkdir(parents=True, exist_ok=True)

license_number = None
try:
    from google.colab import userdata
    try:
        license_number = userdata.get('TOTALSEG_LICENSE')
    except Exception:
        license_number = None
except Exception:
    pass

cmd = [
    'TotalSegmentator',
    '-i', str(INPUT_NII),
    '-o', str(COR_DIR),
    '-ta', 'coronary_arteries',
    '--device', DEVICE,
]
if license_number:
    cmd += ['-l', license_number]
    print('Using TOTALSEG_LICENSE from Colab Secrets (value hidden).')
else:
    print('No TOTALSEG_LICENSE Colab Secret found; trying any existing TotalSegmentator license configuration.')

print('Running coronary-artery segmentation...', flush=True)
t = time.time()
# Do not print the command here because it could include a license argument.
p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
rc_cor = p.returncode
cor_output = p.stdout
print(cor_output[-6000:])
print(f'Coronary runtime/attempt: {(time.time()-t)/60:.1f} min')

CORONARY_AVAILABLE = (rc_cor == 0)
if not CORONARY_AVAILABLE:
    print('\nCORONARY SEGMENTATION NOT AVAILABLE.')
    print('The notebook will continue with aorta-only validation.')
    print('If the message above says a license is required, obtain/configure a TotalSegmentator license and rerun.')
else:
    print('Coronary segmentation completed.')


## Load masks and verify geometry


In [ ]:
def find_mask(folder, preferred):
    exact = folder / preferred
    if exact.exists():
        return exact
    files = sorted(folder.rglob('*.nii.gz'))
    if not files:
        return None
    matches = [p for p in files if preferred.replace('.nii.gz','').lower() in p.name.lower()]
    return matches[0] if matches else files[0]

def read_mask_on_source(path):
    img = sitk.ReadImage(str(path))
    same = (
        img.GetSize() == source_img.GetSize()
        and np.allclose(img.GetSpacing(), source_img.GetSpacing(), atol=1e-5)
        and np.allclose(img.GetOrigin(), source_img.GetOrigin(), atol=1e-4)
        and np.allclose(img.GetDirection(), source_img.GetDirection(), atol=1e-5)
    )
    if not same:
        print('Resampling mask to source geometry:', path.name)
        img = sitk.Resample(
            img, source_img, sitk.Transform(), sitk.sitkNearestNeighbor,
            0, img.GetPixelID()
        )
    return sitk.GetArrayFromImage(img) > 0

aorta_path = find_mask(AORTA_DIR, 'aorta.nii.gz')
if aorta_path is None:
    raise FileNotFoundError('Could not find TotalSegmentator aorta mask.')
aorta = read_mask_on_source(aorta_path)

cor_path = None
cor = None
if CORONARY_AVAILABLE:
    cor_path = find_mask(COR_DIR, 'coronary_arteries.nii.gz')
    if cor_path is None:
        print('Coronary task returned success but no NIfTI mask was found.')
        CORONARY_AVAILABLE = False
    else:
        cor = read_mask_on_source(cor_path)

print('Aorta mask:', aorta_path)
print('Aorta voxels:', int(aorta.sum()))
if CORONARY_AVAILABLE:
    print('Coronary mask:', cor_path)
    print('Coronary voxels:', int(cor.sum()))
    print('Slices containing coronary mask:', int(np.sum(np.any(cor, axis=(1,2)))))


## Static validation gallery

These images are the only outputs needed for the next decision. They show full axial context and close-up overlays. No sliders or manual coordinates are used.


In [ ]:
OUTDIR = ROOT / 'TotalSegmentator_Validation'
OUTDIR.mkdir(parents=True, exist_ok=True)

def diverse_slices_from_counts(counts, n=12, min_sep=6):
    order = np.argsort(counts)[::-1]
    chosen = []
    for z in order:
        if counts[z] <= 0:
            break
        if all(abs(int(z)-p) >= min_sep for p in chosen):
            chosen.append(int(z))
            if len(chosen) >= n:
                break
    return sorted(chosen)

if CORONARY_AVAILABLE:
    counts = cor.sum(axis=(1,2))
    zs = diverse_slices_from_counts(counts, n=12, min_sep=max(4, int(round(1.8/source_img.GetSpacing()[2]))))
else:
    counts = aorta.sum(axis=(1,2))
    zs = diverse_slices_from_counts(counts, n=12, min_sep=max(8, int(round(4.0/source_img.GetSpacing()[2]))))

if len(zs) < 6:
    nz = np.where(counts > 0)[0]
    if len(nz):
        zs = np.linspace(nz.min(), nz.max(), 12).round().astype(int).tolist()

fig, axes = plt.subplots(3,4,figsize=(16,12))
for ax, z in zip(axes.ravel(), zs[:12]):
    ax.imshow(source[z], cmap='gray', vmin=-200, vmax=900)
    try:
        ax.contour(aorta[z].astype(float), levels=[0.5], linewidths=1.2)
    except Exception:
        pass
    if CORONARY_AVAILABLE:
        ax.imshow(np.ma.masked_where(~cor[z], cor[z]), alpha=0.55)
    ax.set_title(f'z={z}')
    ax.axis('off')
plt.tight_layout()
full_path = OUTDIR / 'totalseg_full_context_validation.png'
fig.savefig(full_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved:', full_path)

# Close-ups centered on the aorta/coronary region.
fig, axes = plt.subplots(3,4,figsize=(14,14))
for ax, z in zip(axes.ravel(), zs[:12]):
    target = aorta[z].copy()
    if CORONARY_AVAILABLE:
        target |= cor[z]
    yy, xx = np.where(target)
    if len(xx):
        cy, cx = int(np.median(yy)), int(np.median(xx))
    else:
        cy, cx = source.shape[1]//2, source.shape[2]//2
    r = 110
    y0,y1=max(0,cy-r),min(source.shape[1],cy+r)
    x0,x1=max(0,cx-r),min(source.shape[2],cx+r)
    ax.imshow(source[z,y0:y1,x0:x1], cmap='gray', vmin=-200, vmax=900)
    try:
        ax.contour(aorta[z,y0:y1,x0:x1].astype(float), levels=[0.5], linewidths=1.2)
    except Exception:
        pass
    if CORONARY_AVAILABLE:
        ax.imshow(np.ma.masked_where(~cor[z,y0:y1,x0:x1], cor[z,y0:y1,x0:x1]), alpha=0.60)
    ax.set_title(f'z={z}')
    ax.axis('off')
plt.tight_layout()
close_path = OUTDIR / 'totalseg_aorta_coronary_closeups.png'
fig.savefig(close_path, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved:', close_path)


## Automated sanity checks

These are geometric checks only; they do not assert that the masks are clinically correct.


In [ ]:
spacing = np.array(source_img.GetSpacing(), float)  # x,y,z
voxel_mm3 = float(np.prod(spacing))

rows = [
    {
        'mask':'aorta',
        'voxels':int(aorta.sum()),
        'volume_ml':float(aorta.sum()*voxel_mm3/1000.0),
        'slices':int(np.sum(np.any(aorta,axis=(1,2))))
    }
]
if CORONARY_AVAILABLE:
    rows.append({
        'mask':'coronary_arteries',
        'voxels':int(cor.sum()),
        'volume_ml':float(cor.sum()*voxel_mm3/1000.0),
        'slices':int(np.sum(np.any(cor,axis=(1,2))))
    })

summary = pd.DataFrame(rows)
display(summary)
summary_path = OUTDIR / 'totalseg_mask_summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved:', summary_path)

if CORONARY_AVAILABLE:
    # Contact with aorta: work in a cropped box to avoid allocating a huge distance map.
    zz,yy,xx = np.where(aorta)
    pad_xy = int(round(20.0/min(spacing[:2])))
    pad_z = int(round(20.0/spacing[2]))
    z0,z1=max(0,zz.min()-pad_z),min(aorta.shape[0],zz.max()+pad_z+1)
    y0,y1=max(0,yy.min()-pad_xy),min(aorta.shape[1],yy.max()+pad_xy+1)
    x0,x1=max(0,xx.min()-pad_xy),min(aorta.shape[2],xx.max()+pad_xy+1)

    a_crop = aorta[z0:z1,y0:y1,x0:x1]
    c_crop = cor[z0:z1,y0:y1,x0:x1]
    # approximate 1-2 mm shell with binary dilation, sufficient for validation
    dil = ndi.binary_dilation(a_crop, iterations=4)
    near = c_crop & dil
    print('Coronary voxels within ~few voxels of aorta:', int(near.sum()))
    print('Slices with coronary-aorta proximity:', int(np.sum(np.any(near,axis=(1,2)))))
    if near.sum() == 0:
        print('WARNING: no coronary/aorta proximity was found; inspect overlays carefully.')

print('\nTOTAL SEGMENTATOR VALIDATION COMPLETE.')
print('Please send the two PNG validation images.')
if not CORONARY_AVAILABLE:
    print('Coronary model did not run; send the output text showing the license/model error as well.')
